<a href="https://colab.research.google.com/github/shahwaiz-9/Deep-Learning/blob/main/Binary_Classification_with_RNN_%26_LSTM_%26_GRU_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Dependencies

In [1]:
pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 16.0 MB/s eta 0:00:00


In [2]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 71.6 MB/s eta 0:00:00


In [3]:
import re
import nltk
import contractions
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

In [36]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D, GRU, SimpleRNN
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from gensim.models import Word2Vec

In [5]:
import numpy as np

In [6]:

# Download once (if not already done)
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

#Data Overview

In [7]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [8]:
import os

# List the files in the downloaded dataset directory
print(os.listdir(path))

['IMDB Dataset.csv']


In [9]:
import pandas as pd

df = pd.read_csv(os.path.join(path, 'IMDB Dataset.csv'))
# df = data.sample(25000)
# Display the first 5 rows of the DataFrame
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [10]:
df.shape

(50000, 2)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [12]:
df.duplicated().sum()

np.int64(418)

In [13]:
df = df.drop_duplicates()

In [14]:
df.duplicated().sum()

np.int64(0)

#Text Preprocessing

In [15]:
# Checking balance of classes

df['sentiment'].value_counts()

,count
sentiment,
positive,24884
negative,24698


In [16]:
stopwords = nltk.corpus.stopwords.words('english')
lemmatizer = WordNetLemmatizer()

In [17]:
df['review'].iloc[0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [18]:
def clean_text(text):

  text = re.sub(r'<.*?>', '', text)
  text = re.sub(r'http\S+|www\S+|https\S+', '', text)
  text = contractions.fix(text)
  text = text.lower()
  text = re.sub(r'[^\w\s]', '', text)

  return text


In [19]:
df['clean_review'] = df['review'].apply(clean_text)

In [20]:
df['clean_review'].iloc[0]

'one of the other reviewers has mentioned that after watching just 1 oz episode you will be hooked they are right as this is exactly what happened with methe first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not a show for the faint hearted or timid this show pulls no punches with regards to drugs sex or violence its is hardcore in the classic use of the wordit is called oz as that is the nickname given to the oswald maximum security state penitentary it focuses mainly on emerald city an experimental section of the prison where all the cells have glass fronts and face inwards so privacy is not high on the agenda them city is home to manyaryans muslims gangstas latinos christians italians irish and moreso scuffles death stares dodgy dealings and shady agreements are never far awayi would say the main appeal of the show is due to the fact that it goes where other shows would not dare forget pretty

In [21]:
def get_wordnet_pos(tag):
    """Convert NLTK POS tag to WordNet POS tag"""
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default fallback

In [22]:
def tokenize_lemmitize(text):

  # Word tokenization
  tokens = word_tokenize(text)

  pos_tags = nltk.pos_tag(tokens)

  lemmatized = []

  for word, tag in pos_tags:
    wn_tag = get_wordnet_pos(tag)
    lemma = lemmatizer.lemmatize(word, pos=wn_tag)

    if word not in stopwords and len(lemma) > 1:
      lemmatized.append(lemma)

  return lemmatized


In [23]:
df['processed_review'] = df['clean_review'].apply(tokenize_lemmitize)

In [24]:
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

#Feature Extraction ( vector embeddings )

In [25]:

w2v_model = Word2Vec(
    sentences=df['processed_review'],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

print(f"Word2Vec Vocabulary Size: {len(w2v_model.wv.key_to_index)}")

Word2Vec Vocabulary Size: 71708


In [26]:

# Create a dictionary which contains the word information about dataset
#   {shahwaiz: 3 ( Ouccrence of the word )}

tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['clean_review'])

vocab_size = len(tokenizer.word_index) + 1
print(f"Tokenizer Vocabulary Size: {vocab_size}")





Tokenizer Vocabulary Size: 221318


In [27]:
embedding_dim = 100
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

print("Embedding Matrix Shape:", embedding_matrix.shape)

Embedding Matrix Shape: (221318, 100)


In [28]:
nonzero_elements = np.count_nonzero(np.any(embedding_matrix, axis=1))
print(f"Percentage of words found in Word2Vec: {(nonzero_elements/vocab_size)*100:.2f}%")

Percentage of words found in Word2Vec: 32.08%


#Modelling

1. LSTM

In [29]:

max_len = 200

X_seq = tokenizer.texts_to_sequences(df['clean_review'])

X_padded = pad_sequences(X_seq, maxlen=max_len, padding='post')

X_train, X_test, y_train, y_test = train_test_split(X_padded, df['label'], test_size=0.2, random_state=42)

# Resulting Output: [1, 12, 6, 25]

In [33]:
model_final = Sequential([

    Embedding(input_dim=vocab_size,
              output_dim=embedding_dim,
              weights=[embedding_matrix],
              input_length=max_len,
              trainable=True,
              mask_zero=True),
    SpatialDropout1D(0.4),
    LSTM(128, dropout=0.2, return_sequences=True),
    LSTM(64, dropout=0.2),
    Dense(1, activation='sigmoid')
])

model_final.compile(optimizer='adam',
                    loss='binary_crossentropy',
                    metrics=['accuracy'])

model_final.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │    22,131,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,131,800 (84.43 MB)

 Trainable params: 22,131,800 (84.43 MB)

 Non-trainable params: 0 (0.00 B)

In [34]:
history = model_final.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 34ms/step - accuracy: 0.7673 - loss: 0.4971 - val_accuracy: 0.8268 - val_loss: 0.4911
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.8181 - loss: 0.4090 - val_accuracy: 0.8773 - val_loss: 0.3020
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.8791 - loss: 0.2929 - val_accuracy: 0.8905 - val_loss: 0.2945
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.9060 - loss: 0.2327 - val_accuracy: 0.8824 - val_loss: 0.2781
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.9349 - loss: 0.1715 - val_accuracy: 0.8978 - val_loss: 0.2717
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.9569 - loss: 0.1219 - val_accuracy: 0.8924 - val_loss: 0.2820
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - accuracy: 0.9685 - loss: 0.0899 - val_accuracy: 0.8967 - val_loss: 0.3052
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.9757 - loss: 0.0673 - 

Better then Machine Learning models

2. RNN

In [37]:
rnn_model = Sequential([

    Embedding(input_dim=vocab_size,
              output_dim=embedding_dim,
              weights=[embedding_matrix],
              input_length=max_len,
              trainable=True,
              mask_zero=True),
    SimpleRNN(128, dropout=0.2, return_sequences=True),
    SimpleRNN(64, dropout=0.2),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

rnn_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │    22,131,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,131,800 (84.43 MB)

 Trainable params: 22,131,800 (84.43 MB)

 Non-trainable params: 0 (0.00 B)

In [38]:
rnn_history = rnn_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test))

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 32s 43ms/step - accuracy: 0.6655 - loss: 0.6145 - val_accuracy: 0.6877 - val_loss: 0.5899
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 23s 36ms/step - accuracy: 0.6834 - loss: 0.5909 - val_accuracy: 0.7377 - val_loss: 0.5257
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 23s 36ms/step - accuracy: 0.7237 - loss: 0.5504 - val_accuracy: 0.7710 - val_loss: 0.4883
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 41s 37ms/step - accuracy: 0.7634 - loss: 0.5057 - val_accuracy: 0.7649 - val_loss: 0.5043
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 23s 36ms/step - accuracy: 0.7759 - loss: 0.4790 - val_accuracy: 0.6831 - val_loss: 0.5894
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 23s 36ms/step - accuracy: 0.7746 - loss: 0.4822 - val_accuracy: 0.6638 - val_loss: 0.6394
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 23s 36ms/step - accuracy: 0.8038 - loss: 0.4432 - val_accuracy: 0.5892 - val_loss: 0.6522
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 24s 38ms/step - accuracy: 0.7413 - loss: 0.5225 - 

See after certain iterations the val accuracy drops significantly!

3. GRU

In [41]:
gru_model = Sequential([
        Embedding(input_dim=vocab_size,
              output_dim=embedding_dim,
              weights=[embedding_matrix],
              input_length=max_len,
              trainable=True,
              mask_zero=True),
        SpatialDropout1D(0.4),
        GRU(128, dropout=0.2, return_sequences=True),
        GRU(64, dropout=0.2),
        Dense(1, activation='sigmoid')
])


gru_model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

gru_model.summary()


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │    22,131,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_2             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,131,800 (84.43 MB)

 Trainable params: 22,131,800 (84.43 MB)

 Non-trainable params: 0 (0.00 B)

In [42]:
gru_history = gru_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_data=(X_test, y_test))

Epoch 1/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 28s 40ms/step - accuracy: 0.7532 - loss: 0.5070 - val_accuracy: 0.8032 - val_loss: 0.4392
Epoch 2/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 25s 41ms/step - accuracy: 0.8594 - loss: 0.3284 - val_accuracy: 0.8894 - val_loss: 0.2720
Epoch 3/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 23s 37ms/step - accuracy: 0.8994 - loss: 0.2469 - val_accuracy: 0.8969 - val_loss: 0.2612
Epoch 4/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 23s 37ms/step - accuracy: 0.9295 - loss: 0.1802 - val_accuracy: 0.8927 - val_loss: 0.2838
Epoch 5/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 24s 39ms/step - accuracy: 0.9560 - loss: 0.1203 - val_accuracy: 0.8990 - val_loss: 0.2713
Epoch 6/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 22s 36ms/step - accuracy: 0.9724 - loss: 0.0775 - val_accuracy: 0.8913 - val_loss: 0.3556
Epoch 7/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 21s 34ms/step - accuracy: 0.9836 - loss: 0.0472 - val_accuracy: 0.8814 - val_loss: 0.4187
Epoch 8/10
620/620 ━━━━━━━━━━━━━━━━━━━━ 20s 32ms/step - accuracy: 0.9886 - loss: 0.0316 - 

Since Gru also gives the same accuracy as LSTM so the choice is to use GRU because of less trainable params and less complexity which will save computational cost as well as train time.
Moreover i intentionally trained model upto 10 epochs we can use Early stop to make things optimized.